In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType,TimestampType, FloatType

from pyspark.sql.functions import to_timestamp, date_format, col

from pyspark.sql.functions import *

catalog_name = "openaq"

### Latest

In [0]:
df_latest_silver = spark.table(f"{catalog_name}.bronze.brz_latest")
display(df_latest_silver)

In [0]:
datetime_utc_nulls = df_latest_silver.filter(col("datetime_utc").isNull()).count()
datetime_local_nulls = df_latest_silver.filter(col("datetime_utc").isNull()).count()

print(f"Total nulls in datetime_utc are {datetime_utc_nulls}")
print(f"Total nulls in datetime_local are {datetime_local_nulls}")

In [0]:
df_latest_silver = df_latest_silver.dropna(subset=["datetime_utc","datetime_local"])

row_count = df_latest_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
df_latest_silver = df_latest_silver.withColumn("std_datetime_utc",date_format(to_timestamp("datetime_utc", "yyyy-MM-dd'T'HH:mm:ssX"),"yyyy-MM-dd HH:mm:ss"))

df_latest_silver = df_latest_silver.withColumn("std_datetime_local",date_format(to_timestamp("datetime_local", "yyyy-MM-dd'T'HH:mm:ssXXX"),"yyyy-MM-dd HH:mm:ss"))

display(df_latest_silver)

In [0]:
df_latest_silver = df_latest_silver.drop("datetime_utc").drop("datetime_local")

In [0]:
df_latest_silver.printSchema()

In [0]:
df_latest_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_latest")

### Sensors

In [0]:
df_sensors_silver = spark.table(f"{catalog_name}.bronze.brz_sensors")
display(df_sensors_silver)

In [0]:
df_sensors_silver.columns

In [0]:
datetime_cols= [
    'coverage_from_local',
    'coverage_from_utc',
    'coverage_to_local',
    'coverage_to_utc',
    'datetime_first_local',
    'datetime_first_utc',
    'datetime_last_local',
    'datetime_last_utc',
    'latest_datetime_local',
    'latest_datetime_utc',
]

for cols in datetime_cols:
    df_sensors_silver = df_sensors_silver.withColumn(f"std_{cols}",date_format(to_timestamp(cols, "yyyy-MM-dd'T'HH:mm:ssXXX"),"yyyy-MM-dd HH:mm:ss"))
    df_sensors_silver = df_sensors_silver.drop(cols)

display(df_sensors_silver)


In [0]:
df_sensors_silver.groupby("coverage_expected_interval").agg(count("*")).show()

In [0]:
df_sensors_silver = df_sensors_silver.withColumn(
    "std_coverage_expected_interval",
    concat(
        when(split(col("coverage_expected_interval"), ":")[0].cast("int") > 0,
             concat(split(col("coverage_expected_interval"), ":")[0].cast("int"), lit(" hr "))
        ).otherwise(lit("")),
        when(split(col("coverage_expected_interval"), ":")[1].cast("int") > 0,
             concat(split(col("coverage_expected_interval"), ":")[1].cast("int"), lit(" min"))
        ).otherwise(lit(""))
    )
)

df_sensors_silver = df_sensors_silver.drop("coverage_expected_interval")

display(df_sensors_silver)

In [0]:
df_sensors_silver = df_sensors_silver.withColumn(
    "std_coverage_observed_interval",
    concat(
        when(split(col("coverage_observed_interval"), ":")[0].cast("int") > 0,
             concat(split(col("coverage_observed_interval"), ":")[0].cast("int"), lit(" hr "))
        ).otherwise(lit("")),
        when(split(col("coverage_observed_interval"), ":")[1].cast("int") > 0,
             concat(split(col("coverage_observed_interval"), ":")[1].cast("int"), lit(" min"))
        ).otherwise(lit(""))
    )
)

df_sensors_silver = df_sensors_silver.drop("coverage_observed_interval")

df_sensors_silver.groupby("std_coverage_observed_interval").agg(count("*")).show()

In [0]:
percent_coverage_columns = ["coverage_percent_complete","coverage_percent_coverage"]

for cols in percent_coverage_columns:
    df_sensors_silver = df_sensors_silver.withColumn(
        f"std_{cols}",
        concat(
            round(
                when(col(f"{cols}")>100000,col(f"{cols}")/100000)
                .when(col(f"{cols}")>10000,col(f"{cols}")/10000)
                .when(col(f"{cols}")>1000,col(f"{cols}")/1000)
                .when(col(f"{cols}")>100,col(f"{cols}")/100)
                .otherwise(col(f"{cols}")) ,2
                ),lit("%")))
    df_sensors_silver = df_sensors_silver.drop(cols)

display(df_sensors_silver)


In [0]:
unnecessary_columns = ['summary_q02',
 'summary_q25',
 'summary_q75',
 'summary_q98',
 'summary_sd']

for cols in unnecessary_columns:
    df_sensors_silver = df_sensors_silver.drop(cols)

In [0]:
df_sensors_silver = df_sensors_silver.withColumn("summary_avg",round(col("summary_avg"),1))
df_sensors_silver.groupby("summary_avg").agg(count("*")).show()

In [0]:
df_sensors_silver.groupby("parameter_name").agg(count("*")).show()

In [0]:
df_sensors_silver = df_sensors_silver.withColumnRenamed("parameter_name","parameter")

In [0]:
parameter_map = {
    "so2": "Sulfur Dioxide",
    "pm10": "Particulate Matter",
    "no2": "Nitrogen Dioxide",
    "co": "Carbon Monoxide",
    "pm25": "Particulate Matter",
    "bc": "Black Carbon",
    "o3": "Ozone"
}

df_sensors_silver = df_sensors_silver.withColumn("std_parameter_name", lower(col("parameter"))).replace(parameter_map, subset=["std_parameter_name"])

display(df_sensors_silver)

In [0]:
df_sensors_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_sensors")